In [2]:
import pandas as pd
import os
os.environ.pop("SSLKEYLOGFILE", None)

url = "https://statistik.leipzig.de/opendata/api/kdvalues?kategorie_nr=2&rubrik_nr=9&periode=y&format=csv"
dichte = pd.read_csv(url, decimal=",")

# Wide -> long (Jahre als Zeilen)
dichte_long = dichte.melt(
    id_vars=["Gebiet", "Sachmerkmal"],
    var_name="jahr",
    value_name="wert"
)


In [3]:
dichte_long.head()       # erste 5 Zeilen
dichte_long.info()       # Spalten, Datentypen, fehlende Werte
dichte_long.describe()   # Kennzahlen für numerische Spalten
dichte_long["Gebiet"].unique()   # alle Gebietsnamen

<class 'pandas.DataFrame'>
RangeIndex: 1628 entries, 0 to 1627
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Gebiet       1628 non-null   str    
 1   Sachmerkmal  1628 non-null   str    
 2   jahr         1628 non-null   str    
 3   wert         1628 non-null   float64
dtypes: float64(1), str(3)
memory usage: 51.0 KB


<StringArray>
[                    'Zentrum',                 'Zentrum-Ost',
              'Zentrum-Südost',                 'Zentrum-Süd',
                'Zentrum-West',            'Zentrum-Nordwest',
                'Zentrum-Nord',      'Schönefeld-Abtnaundorf',
              'Schönefeld-Ost',                  'Mockau-Süd',
                 'Mockau-Nord',                      'Thekla',
             'Plaußig-Portitz',      'Neustadt-Neuschönefeld',
                'Volkmarsdorf',           'Anger-Crottendorf',
          'Sellerhausen-Stünz',                   'Paunsdorf',
                 'Heiterblick',                      'Mölkau',
                  'Engelsdorf',                   'Baalsdorf',
           'Althen-Kleinpösna',           'Reudnitz-Thonberg',
                  'Stötteritz',                 'Probstheida',
                    'Meusdorf',             'Liebertwolkwitz',
                  'Holzhausen',                 'Südvorstadt',
                   'Connewitz',          

In [4]:
print(1+1)
print(dichte_long.head())

2
           Gebiet      Sachmerkmal  jahr    wert
0         Zentrum  Einwohnerdichte  2004  2951.0
1     Zentrum-Ost  Einwohnerdichte  2004  2066.0
2  Zentrum-Südost  Einwohnerdichte  2004  4015.0
3     Zentrum-Süd  Einwohnerdichte  2004  6784.0
4    Zentrum-West  Einwohnerdichte  2004  5763.0


In [ ]:
import matplotlib.pyplot as plt

# Arbeitskopie für EDA
eda = dichte_long.copy()
eda["jahr"] = pd.to_numeric(eda["jahr"])
eda = eda.sort_values("jahr")

latest_year = eda["jahr"].max()
latest = eda[eda["jahr"] == latest_year].sort_values("wert", ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Gebiete mit höchster Einwohnerdichte im letzten Jahr
top = latest.head(15).sort_values("wert")
axes[0, 0].barh(top["Gebiet"], top["wert"], color="steelblue")
axes[0, 0].set_title(f"Top 15 Einwohnerdichte ({latest_year})")
axes[0, 0].set_xlabel("Einwohner je km²")

# 2. Verteilung der Einwohnerdichte im letzten Jahr
axes[0, 1].hist(latest["wert"], bins=15, color="darkorange", edgecolor="white")
axes[0, 1].set_title(f"Verteilung der Einwohnerdichte ({latest_year})")
axes[0, 1].set_xlabel("Einwohner je km²")
axes[0, 1].set_ylabel("Anzahl Gebiete")

# 3. Zeitliche Entwicklung ausgewählter Gebiete
auswahl = [
    "Stadt Leipzig", "Zentrum", "Zentrum-Süd",
    "Zentrum-West", "West", "Nord"
]
auswahl = [gebiet for gebiet in auswahl if gebiet in eda["Gebiet"].unique()]

for gebiet in auswahl:
    daten = eda[eda["Gebiet"] == gebiet]
    axes[1, 0].plot(daten["jahr"], daten["wert"], marker="o", label=gebiet)

axes[1, 0].set_title("Entwicklung ausgewählter Gebiete")
axes[1, 0].set_xlabel("Jahr")
axes[1, 0].set_ylabel("Einwohner je km²")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Entwicklung der Streuung über die Jahre
jahreswerte = [
    eda.loc[eda["jahr"] == jahr, "wert"].dropna()
    for jahr in sorted(eda["jahr"].unique())
]
jahre = sorted(eda["jahr"].unique())

axes[1, 1].boxplot(jahreswerte, labels=jahre, showfliers=False)
axes[1, 1].set_title("Streuung der Einwohnerdichte nach Jahr")
axes[1, 1].set_xlabel("Jahr")
axes[1, 1].set_ylabel("Einwohner je km²")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()